# 02 — Pre-ingest checks: tile variability, edge filtering, dedupe, crop codes

Notebook 01 profiled one interior tile. Before running all 196 tiles, we close the
open questions that would change the design of the ingest pipeline.

| # | Question | Pipeline decision it drives |
|---|---|---|
| 1 | How much do parcel counts vary across tiles? | pagination, timeouts, runtime estimate |
| 2 | Does department filtering work on an edge tile? | membership rule for parcels crossing the boundary |
| 3 | Are parcels on shared tile edges returned identically by both tiles? | whether dedupe on `id_parcel` is safe |
| 4 | What does `code_group == "20"` contain? Where is kiwi? | definition of "pollination target parcel" |

Acceptance criteria are written **before** running, so the results can't bend them.

In [ ]:
from __future__ import annotations

import json
import re
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from shapely.geometry import box, mapping
from urllib3.util.retry import Retry

DEPT_CODE = "82"
RPG_YEAR = 2024
CRS_WGS84 = "EPSG:4326"
CRS_L93 = "EPSG:2154"
GEO_API = "https://geo.api.gouv.fr"
RPG_API = "https://apicarto.ign.fr/api/rpg/v2"
CROP_CODES_URL = (
    "https://static.data.gouv.fr/resources/codes-cultures-pac/"
    "20240402-091659/n-cultures-2024.ods"
)
TILE_SIZE_M = 5000
PAGE_SIZE = 1000
N_RANDOM_TILES = 5
SEED = 42  # fixed seed -> the same "random" tiles on every rerun

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE_DIR = ROOT / "data" / "raw" / "profiling" / f"rpg_{RPG_YEAR}"   # gitignored
REF_DIR = ROOT / "data" / "raw" / "reference"                        # gitignored
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REF_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

In [ ]:
def make_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=1.0, status_forcelist=(429, 500, 502, 503, 504))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def get_json(session, url, params, timeout=120) -> dict:
    r = session.get(url, params=params, timeout=timeout)
    if not r.ok:
        print("HTTP", r.status_code, r.text[:1000])  # show the error body before raising
        r.raise_for_status()
    return r.json()


def fetch_rpg_tile(session, tile_id: int, geom) -> dict:
    """Fetch all pages for one tile. Raw responses are cached on disk, so reruns cost nothing."""
    path = CACHE_DIR / f"tile_{tile_id:04d}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8")) | {"from_cache": True}

    features, pages, start, t0 = [], 0, 0, time.perf_counter()
    while True:
        params = {"annee": RPG_YEAR, "geom": json.dumps(mapping(geom)),
                  "_limit": PAGE_SIZE, "_start": start}
        batch = get_json(session, RPG_API, params).get("features", [])
        features.extend(batch)
        pages += 1
        if len(batch) < PAGE_SIZE:
            break
        start += PAGE_SIZE

    result = {"tile_id": tile_id, "pages": pages,
              "seconds": round(time.perf_counter() - t0, 2), "features": features}
    path.write_text(json.dumps(result), encoding="utf-8")
    return result | {"from_cache": False}


def to_gdf(result: dict) -> gpd.GeoDataFrame:
    gdf = gpd.GeoDataFrame.from_features(result["features"], crs=CRS_WGS84)
    gdf["tile_id"] = result["tile_id"]
    return gdf


session = make_session()

## Setup — department boundary and tile grid (same logic as notebook 01)

In [ ]:
communes_raw = get_json(
    session,
    f"{GEO_API}/departements/{DEPT_CODE}/communes",
    {"format": "geojson", "geometry": "contour", "fields": "code,nom,population"},
)
communes = gpd.GeoDataFrame.from_features(communes_raw["features"], crs=CRS_WGS84)
boundary_l93 = communes.to_crs(CRS_L93).union_all()

minx, miny, maxx, maxy = boundary_l93.bounds
grid = gpd.GeoDataFrame(
    geometry=[
        box(x, y, x + TILE_SIZE_M, y + TILE_SIZE_M)
        for x in np.arange(minx, maxx, TILE_SIZE_M)
        for y in np.arange(miny, maxy, TILE_SIZE_M)
    ],
    crs=CRS_L93,
)
grid = grid[grid.intersects(boundary_l93)].reset_index(drop=True)
grid["tile_id"] = grid.index
grid["inside_ratio"] = grid.intersection(boundary_l93).area / grid.area
grid_wgs = grid.to_crs(CRS_WGS84)

print(f"{len(grid)} tiles")
print(grid["inside_ratio"].describe().round(2))
print("interior tiles (100% inside):", (grid["inside_ratio"] > 0.999).sum())

## Check 1 — Parcel count variability across 5 random tiles

**Acceptance criteria**
- Record parcels, pages, response size and time per tile.
- If any tile needs more than 1 page, pagination is confirmed as mandatory (not optional).
- Produce a full-run estimate from the mean, with a range from min/max.

In [ ]:
rng = np.random.default_rng(SEED)
random_ids = sorted(rng.choice(grid["tile_id"].to_numpy(), size=N_RANDOM_TILES, replace=False).tolist())

rows, random_gdfs = [], []
for tid in random_ids:
    res = fetch_rpg_tile(session, tid, grid_wgs.geometry.iloc[tid])
    rows.append({
        "tile_id": tid,
        "inside_ratio": round(grid.loc[tid, "inside_ratio"], 2),
        "parcels": len(res["features"]),
        "pages": res["pages"],
        "kb": round(len(json.dumps(res["features"]).encode()) / 1024),
        "seconds": res["seconds"],
        "from_cache": res["from_cache"],
    })
    random_gdfs.append(to_gdf(res))

variability = pd.DataFrame(rows)
display(variability)
print(variability[["parcels", "kb", "seconds"]].describe().round(1))

In [ ]:
n = len(grid)
p = variability["parcels"]
s = variability["seconds"]
kb = variability["kb"]
print(f"Full-run estimate over {n} tiles:")
print(f"  parcels : ~{p.mean() * n:,.0f}   (range {p.min() * n:,.0f} - {p.max() * n:,.0f})")
print(f"  size    : ~{kb.mean() * n / 1024:,.0f} MB raw JSON")
print(f"  runtime : ~{s.mean() * n / 60:,.1f} min sequential (cached tiles show 0 s)")
print("  max pages per tile:", variability["pages"].max())

In [ ]:
# Partial view of crop groups across the 5 tiles (full answer comes after the full ingest)
sample = pd.concat(random_gdfs, ignore_index=True)
print("code_group (5 random tiles):")
print(sample["code_group"].value_counts().head(20))
print("\nnull share per column:")
print(sample.drop(columns="geometry").isna().mean().round(3))

## Check 2 — Department filtering on an edge tile

The RPG API returns every parcel intersecting the query box, including parcels in the
neighbouring departments. Each parcel falls in one of three classes:

- **inside**: entirely within the department
- **straddling**: crosses the department boundary
- **outside**: does not touch the department

`intersects` keeps inside + straddling. A stricter rule is
"the parcel's `representative_point()` lies inside the department": every parcel then
belongs to exactly one department, which matters once neighbouring departments are ingested too.
(`representative_point` is guaranteed to be inside the polygon; `centroid` is not.)

**Acceptance criteria**
- The edge tile returns parcels in all three classes.
- No invalid geometry reaches the spatial predicates (make_valid first).
- We pick a membership rule and write down why.

In [ ]:
# Edge tile: the one whose inside ratio is closest to 50%
edge_id = int((grid["inside_ratio"] - 0.5).abs().idxmin())
print(f"edge tile_id={edge_id}, inside_ratio={grid.loc[edge_id, 'inside_ratio']:.2f}")

edge_res = fetch_rpg_tile(session, edge_id, grid_wgs.geometry.iloc[edge_id])
edge = to_gdf(edge_res).to_crs(CRS_L93)
print("parcels returned:", len(edge))

# Spatial predicates are only reliable on valid geometry -> fix before classifying
print("invalid before make_valid:", (~edge.is_valid).sum())
edge["geometry"] = edge.geometry.make_valid()

In [ ]:
within = edge.within(boundary_l93)
intersects = edge.intersects(boundary_l93)
edge["location"] = np.select([within, intersects], ["inside", "straddling"], default="outside")
edge["rep_point_inside"] = edge.representative_point().within(boundary_l93)

print(edge["location"].value_counts())
print("\nkept by 'intersects'          :", intersects.sum())
print("kept by 'representative point' :", edge["rep_point_inside"].sum())
print("\nstraddling parcels split by representative point:")
print(edge.loc[edge["location"] == "straddling", "rep_point_inside"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
colors = {"inside": "tab:green", "straddling": "tab:orange", "outside": "tab:red"}
for loc, color in colors.items():
    part = edge[edge["location"] == loc]
    if len(part):
        part.plot(ax=ax, color=color, linewidth=0, label=f"{loc} ({len(part)})")
gpd.GeoSeries([boundary_l93], crs=CRS_L93).boundary.plot(ax=ax, color="black", linewidth=1)
gpd.GeoSeries([grid.geometry.iloc[edge_id]], crs=CRS_L93).boundary.plot(ax=ax, color="grey", linestyle="--")
ax.set_xlim(*grid.geometry.iloc[edge_id].buffer(500).bounds[0::2])
ax.set_ylim(*grid.geometry.iloc[edge_id].buffer(500).bounds[1::2])
ax.legend(loc="lower left")
ax.set_title(f"Edge tile {edge_id}: parcels by location vs department boundary")
ax.set_axis_off()
plt.show()

## Check 3 — Are duplicates across neighbouring tiles identical?

A parcel crossing the line between two tiles is returned by both queries. Dedupe on
`id_parcel` with "keep first" is only safe if both copies are **identical**. If the API
clipped geometries to the query box, the two copies would be different halves, and
dedupe would silently drop half of the parcel.

**Acceptance criteria**
- At least one shared `id_parcel` between the edge tile and its neighbour.
- 100% of shared parcels have equal attributes and equal geometry (within 1 cm).

In [ ]:
# Neighbour: the tile sharing the longest border with the edge tile
edge_box = grid.geometry.iloc[edge_id]
shared_len = grid.geometry.intersection(edge_box).length
shared_len[edge_id] = 0
neighbour_id = int(shared_len.idxmax())
print(f"neighbour tile_id={neighbour_id} (shared border {shared_len.max():.0f} m)")

nb = to_gdf(fetch_rpg_tile(session, neighbour_id, grid_wgs.geometry.iloc[neighbour_id])).to_crs(CRS_L93)
a = to_gdf(edge_res).to_crs(CRS_L93).set_index("id_parcel")  # raw copy, before make_valid
b = nb.set_index("id_parcel")

shared = a.index.intersection(b.index)
print("shared id_parcel:", len(shared))

attrs = ["surf_parc", "code_cultu", "code_group", "cat_cult_p"]
same_attrs = (a.loc[shared, attrs].fillna("<null>") == b.loc[shared, attrs].fillna("<null>")).all(axis=1)
same_geom = a.loc[shared].geometry.geom_equals_exact(b.loc[shared].geometry, tolerance=0.01)
print(f"identical attributes: {same_attrs.mean():.0%}")
print(f"identical geometry  : {same_geom.mean():.0%}")
if len(shared) and not same_geom.all():
    diff = (a.loc[shared].area - b.loc[shared].area).abs()
    print("area difference (m2) on non-identical pairs:\n", diff[~same_geom].describe())

## Check 4 — What do the crop codes mean? (reference data, not sampling)

Sampling tiles cannot tell us what `code_group == "20"` *means*; only the official
nomenclature can. The PAC crop code list is published on data.gouv.fr
("Codes cultures PAC", one ODS file per year).

The file is a transcription of a PDF notice, so its header layout is unknown in advance.
Instead of guessing column names, we search **cell values**.
Reading `.ods` needs `odfpy` (`conda install -c conda-forge odfpy`).

In [ ]:
ref_path = REF_DIR / "n-cultures-2024.ods"
if not ref_path.exists():
    r = session.get(CROP_CODES_URL, timeout=120)
    r.raise_for_status()
    ref_path.write_bytes(r.content)

sheets = pd.read_excel(ref_path, engine="odf", sheet_name=None, header=None, dtype=str)
for name, df in sheets.items():
    print(f"sheet '{name}': {df.shape}")
first_sheet = next(iter(sheets.values()))
display(first_sheet.head(12))

In [ ]:
def find_rows(pattern: str) -> pd.DataFrame:
    """Return every row (any sheet) where any cell matches the regex, case-insensitive."""
    hits = []
    for name, df in sheets.items():
        mask = df.apply(lambda col: col.str.contains(pattern, case=False, na=False, regex=True)).any(axis=1)
        if mask.any():
            hits.append(df[mask].assign(sheet=name))
    return pd.concat(hits) if hits else pd.DataFrame()


print("Rows mentioning kiwi:")
display(find_rows(r"kiwi|actinidia"))

print("Rows mentioning orchards / apple:")
display(find_rows(r"verger|pomm"))

In [ ]:
# Which codes from our 5-tile sample fall in group 20, and what are they?
codes_20 = sorted(sample.loc[sample["code_group"] == "20", "code_cultu"].unique())
print("code_cultu observed with code_group 20:", codes_20)
for c in codes_20:
    display(find_rows(rf"^{re.escape(c)}$"))

## Findings (fill in after running)

- **Check 1:** parcels per tile min / median / max = … ; max pages = … ; full run ≈ … min, … MB.
- **Check 2:** inside / straddling / outside = … / … / … ; chosen membership rule: … because …
- **Check 3:** shared ids = … ; identical = …% → dedupe on `id_parcel` is / is not safe.
- **Check 4:** group 20 contains … ; kiwi code is … in group … → pollination target definition: …